In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import os
import itertools
from scipy.spatial import KDTree

# --- FUNCIONES DE APOYO ---

def rgb_to_hex(color):
    return "#{:02x}{:02x}{:02x}".format(int(color[0]), int(color[1]), int(color[2]))

def hex_to_rgb(hex_code):
    hex_code = hex_code.lstrip('#')
    return tuple(int(hex_code[i:i+2], 16) for i in (0, 2, 4))

def get_text_color(rgb):
    luminance = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]
    return (255, 255, 255) if luminance < 128 else (0, 0, 0)

def get_exact_color_counts(image_path, top_n=20):
    img = cv2.imread(image_path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pixels = img.reshape(-1, 3)
    counts = Counter([tuple(p) for p in pixels])
    total_pixels = len(pixels)
    sorted_counts = counts.most_common(top_n)

    data_list = []
    for i, (color, count) in enumerate(sorted_counts):
        data_list.append({'ID': i, 'HEX': rgb_to_hex(color), 'Porcentaje': (count/total_pixels)*100, 'RGB': color})
    return pd.DataFrame(data_list)

def process_batch_automated(or_dir, df_colores, similar_colors, replace_dict, folder_name):
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    palette_rgb = np.array(df_colores['RGB'].tolist())
    tree = KDTree(palette_rgb)

    mapping_rgb = {}
    for group_idx, target_hex in replace_dict.items():
        new_rgb = hex_to_rgb(target_hex)
        for color_id in similar_colors[group_idx]:
            row = df_colores.loc[df_colores['ID'] == color_id]
            if not row.empty:
                mapping_rgb[tuple(row['RGB'].values[0])] = new_rgb

    for img_name in os.listdir(or_dir):
        if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')): continue

        img = cv2.imread(os.path.join(or_dir, img_name), cv2.IMREAD_UNCHANGED)
        if img is None: continue

        bgr = img[:, :, :3]
        alpha = img[:, :, 3] if img.shape[2] == 4 else None
        rgb_img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

        h, w, c = rgb_img.shape
        pixels_flat = rgb_img.reshape(-1, 3)
        _, indices = tree.query(pixels_flat)
        rgb_img_quantized = palette_rgb[indices].reshape(h, w, c)

        new_rgb_img = rgb_img_quantized.copy()
        for orig_rgb, target_rgb in mapping_rgb.items():
            mask = np.all(rgb_img_quantized == orig_rgb, axis=-1)
            new_rgb_img[mask] = target_rgb

        final_bgr = cv2.cvtColor(new_rgb_img.astype(np.uint8), cv2.COLOR_RGB2BGR)
        final_img = cv2.merge([final_bgr, alpha]) if alpha is not None else final_bgr

        cv2.imwrite(os.path.join(folder_name, img_name), final_img)

ModuleNotFoundError: No module named 'cv2'

In [ ]:
# --- PALETA BASE (extraída de nudge1.png, misma lógica que change_colors_5) ---
#
# ID  0 | #ffffff  -> fondo blanco         (NO se mapea)
# ID  1 | #6cdb30  -> PIEL (verde base)
# ID  2 | #db4930  -> MANCHAS (rojo base)
# ID  3 | #30dbd3  -> OREJAS (cyan base)
# ID  4 | #000000  -> BORDES (negro base)
# ID  5 | #6cda30  -> piel variación
# ID  6 | #b3220a  -> manchas variación oscura
# ID  7 | #010101  -> bordes variación
# IDs 8,11,14,15,18 -> blanco variaciones (fondo, NO se mapean)
# IDs 9,10,12,13,16,17,19 -> piel variaciones

df_colores = get_exact_color_counts('./nudge1.png', top_n=20)
print(df_colores.to_string())

In [ ]:
# --- OPCIONES DE COLOR (los mismos que change_colors_5_new_combs) ---

piel_opts    = ["#B6975B", "#191919", "#D28C46", "#F0F0EB"]  # Naranja, Gris oscuro, Naranja claro, Beige
manchas_opts = ["#824B28", "#2D2D2D", "#DCDCD7", "#946A2D"]  # Marrón, Gris oscuro, Blanco, Marrón claro
orejas_opts  = ["#E1B9B4", "#FCE5CD"]                         # Rosado, Crema

# Agrupación de IDs de nudge1.png
similar_colors = [
    [1, 5, 9, 10, 12, 13, 16, 17, 19],  # Grupo 0 -> Piel
    [4, 7],                              # Grupo 1 -> Bordes (color fijo)
    [2, 6],                              # Grupo 2 -> Manchas
    [3],                                 # Grupo 3 -> Orejas
]

# --- EJECUCIÓN PRINCIPAL ---

combinations = list(itertools.product(
    range(len(piel_opts)),
    range(len(manchas_opts)),
    range(len(orejas_opts)),
))

print(f"[*] Iniciando generación de {len(combinations)} variantes...")

for combo in combinations:
    p, m, e = combo
    folder_id = f"{p}{m}{e}"
    folder_name = f"nudge_{folder_id}"

    replace_dict = {
        0: piel_opts[p],
        1: "#2C3E50",       # Bordes: color oscuro fijo
        2: manchas_opts[m],
        3: orejas_opts[e],
    }

    process_batch_automated("./", df_colores, similar_colors, replace_dict, folder_name)
    print(f"[+] Generada: {folder_name}")

print("\n[!] Proceso finalizado. Revisá las carpetas generadas.")